

---

# 📘 LeetCode 1892: Page Recommendations II

**Level:** Hard  

---

## ❓ Question

You are implementing a page recommendation system for a social media website.  

A page will be recommended to a user if:  
- The page is liked by at least one of their friends.  
- The user has not already liked the page.  

Return all possible page recommendations with the following columns:  
- `user_id`: The ID of the user receiving the recommendation.  
- `page_id`: The ID of the recommended page.  
- `friends_likes`: The number of friends of `user_id` who like `page_id`.  

---

## 📊 Sample Data

### Friendship Table

| user1_id | user2_id |
|----------|----------|
| 1        | 2        |
| 1        | 3        |
| 1        | 4        |
| 2        | 3        |
| 2        | 4        |
| 2        | 5        |
| 6        | 1        |

### Likes Table

| user_id | page_id |
|---------|---------|
| 1       | 88      |
| 2       | 23      |
| 3       | 24      |
| 4       | 56      |
| 5       | 11      |
| 6       | 33      |
| 2       | 77      |
| 3       | 77      |
| 6       | 88      |

---

## 🏗️ Schema Definition

```python
from pyspark.sql.types import StructType, StructField, IntegerType

# Friendship schema
friendship_schema = StructType([
    StructField("user1_id", IntegerType(), False),
    StructField("user2_id", IntegerType(), False)
])

# Likes schema
likes_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("page_id", IntegerType(), False)
])
```

---

## 📥 Data Preparation

```python
# Friendship data
friendship_data = [
    (1, 2),
    (1, 3),
    (1, 4),
    (2, 3),
    (2, 4),
    (2, 5),
    (6, 1)
]

# Likes data
likes_data = [
    (1, 88),
    (2, 23),
    (3, 24),
    (4, 56),
    (5, 11),
    (6, 33),
    (2, 77),
    (3, 77),
    (6, 88)
]
```

---

## 🗂️ Create DataFrames

```python
# Create Friendship DataFrame
friendship_df = spark.createDataFrame(friendship_data, schema=friendship_schema)
friendship_df.show()

# Create Likes DataFrame
likes_df = spark.createDataFrame(likes_data, schema=likes_schema)
likes_df.show()
```

---

## 👁️ Register as SQL Views

```python
friendship_df.createOrReplaceTempView("Friendship")
likes_df.createOrReplaceTempView("Likes")
```

---


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Friendship schema
friendship_schema = StructType([
    StructField("user1_id", IntegerType(), False),
    StructField("user2_id", IntegerType(), False)
])

# Likes schema
likes_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("page_id", IntegerType(), False)
])
# Friendship data
friendship_data = [
    (1, 2),
    (1, 3),
    (1, 4),
    (2, 3),
    (2, 4),
    (2, 5),
    (6, 1)
]

# Likes data
likes_data = [
    (1, 88),
    (2, 23),
    (3, 24),
    (4, 56),
    (5, 11),
    (6, 33),
    (2, 77),
    (3, 77),
    (6, 88)
]
# Create Friendship DataFrame
friendship_df = spark.createDataFrame(friendship_data, schema=friendship_schema)
friendship_df.show()

# Create Likes DataFrame
likes_df = spark.createDataFrame(likes_data, schema=likes_schema)
likes_df.show()

friendship_df.createOrReplaceTempView("friendship")
likes_df.createOrReplaceTempView("likes")

In [0]:
%sql
WITH cte AS (
		SELECT user1_id AS SELF,
			user2_id AS friend
		FROM Friendship
		
		UNION
		
		SELECT user2_id AS SELF,
			user1_id AS friend
		FROM friendship
		)

SELECT SELF AS user_id,
	page_id,
	count(friend) friends_likes
FROM cte
LEFT JOIN likes
	ON cte.friend = likes.user_id
WHERE page_id NOT IN (
		SELECT DISTINCT page_id
		FROM likes il
		WHERE user_id = cte.SELF
		)
GROUP BY SELF,
	page_id
ORDER BY SELF ASC
